In [ ]:
import re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

def aplicar_bag_of_words_porcentagem(df, df_regex, corte_porcentagem=70.0):
    df = df.copy()
    df["temas_encontrados"] = [[] for _ in range(len(df))]
    df["termos_capturados"] = [[] for _ in range(len(df))]
    
    # Garantir que o texto seja string e esteja em minúsculo
    df["texto_processado"] = df["inteiro_teor_lematizado"].fillna("").astype(str).str.lower()

    for _, row in df_regex.iterrows():
        regex_completa = row["REGEX"]
        tema_codigo = row["TEMA CÓDIGO"]

        if regex_completa == "NAN" or pd.isna(regex_completa):
            continue

        # 1. Extrair os blocos e limpá-los para virarem palavras puras (Bag of Words)
        blocos_sujos = re.findall(r"\(([^)]+)\)", regex_completa)
        
        categorias_palavras = []
        todas_palavras_do_tema = []
        
        for bloco in blocos_sujos:
            # Limpa escapes de regex (\b, \., \ ) e divide pelos "OU" (|)
            termos = [t.replace(r'\b', '').replace(r'\.', '.').replace('\\', '').strip().lower() 
                      for t in bloco.split('|')]
            # Filtra termos vazios
            termos = [t for t in termos if t]
            if termos:
                categorias_palavras.append(termos)
                todas_palavras_do_tema.extend(termos)

        total_categorias = len(categorias_palavras)
        if total_categorias == 0:
            continue

        print(f"Processando Tema {tema_codigo} via BoW ({total_categorias} categorias mapeadas)...")

        # 2. Criar o vetorizador de Bag of Words apenas para as palavras deste tema
        # Usamos ngram_range=(1, 4) para capturar termos compostos como "ação civil pública"
        vectorizer = CountVectorizer(vocabulary=list(set(todas_palavras_do_tema)), ngram_range=(1, 4), lower_case=True)
        
        try:
            X = vectorizer.transform(df["texto_processado"])
            recursos = vectorizer.get_feature_names_out()
        except Exception:
            # Caso o vocabulário esteja vazio ou falhe, pula para o próximo
            continue

        # Matriz booleana: Indica True se a palavra do vocabulário existe no documento
        matriz_presenca = X.toarray() > 0

        # 3. Verificar linha por linha do DataFrame
        for idx in range(len(df)):
            # Descobre quais palavras do vocabulário geral estão nesta linha específica
            palavras_presentes_na_linha = set(recursos[matriz_presenca[idx]])
            
            if not palavras_presentes_na_linha:
                continue

            categorias_atendidas = 0
            termos_encontrados = []

            # Verifica, para cada uma das 23 categorias, se pelo menos uma palavra dela apareceu
            for lista_categoria in categorias_palavras:
                intersecao = palavras_presentes_na_linha.intersection(lista_categoria)
                if intersecao:
                    categorias_atendidas += 1
                    termos_encontrados.extend(list(intersecao))

            # 4. Cálculo do Score
            porcentagem_sucesso = (categorias_atendidas / total_categorias) * 100

            if porcentagem_sucesso >= corte_porcentagem:
                df.at[df.index[idx], "temas_encontrados"].append(f"{tema_codigo} ({porcentagem_sucesso:.1f}%)")
                df.at[df.index[idx], "termos_capturados"].append(f"{tema_codigo}: {list(set(termos_encontrados))}")

    # Remove a coluna temporária de processamento
    df.drop(columns=["texto_processado"], inplace=True)
    return df

In [ ]:
df_lematizado = pd.read_parquet(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification-1\data\dataset_enriquecido_10062026_lematizado.parquet")
df_temas = pd.read_csv(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification-1\data\Temas_STF_com_Regex.csv")

In [ ]:
df_lematizado_sample = df_lematizado.sample(frac=0.01, random_state=42)
df_sample = df_lematizado_sample.iloc[[0]]

In [ ]:
# Chame a função passando os seus DataFrames
df_resultado = aplicar_bag_of_words_porcentagem(
    df=df_lematizado_sample, 
    df_regex=df_temas, 
    corte_porcentagem=70.0  # Você pode mudar para 60.0, 80.0, etc.
)

In [ ]:
df_resultado.head()